In [1]:
import sys
from pathlib import Path
# Get project root (current working directory when running from project root)
# If running from research/, use parent; otherwise use cwd()
project_root = Path.cwd() if (Path.cwd() / 'vault').exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

In [2]:
from utils.enums import Ticker, TimeFrame, Direction
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from feature_extraction.feature_extractor import extract_features
from feature_selection.base_models import BaseModel, QuantileBinningModel
from ensemble.vault_manager import (
    initialize_vault, create_ensemble_directory, 
    load_feature_base_models, list_features
)
from eda.feature_explorer import FeatureExplorer
import utils.helpers as helpers

/home/raman/repos/Trading-Algo/utils/permutation_test/permute_bars.py:38: UserWarning: feature_selection.opt_thresh not found. Using simplified threshold optimization. For full functionality, ensure the opt_thresh module is available.
  warnings.warn(


# Buy and Hold Bias Node - Vault Integration Test

This notebook demonstrates:
1. Extracting the buy_hold feature (always outputs 1.0)
2. Visualizing the feature and comparing with returns
3. Creating a BaseModel with binning
4. Saving to vault as an end-to-end integration test

In [3]:
# Step 1: Load candles data for multiple tickers
# BaseModel will compute features internally from candles
# Use the new multi-ticker helper that appends rows with ticker column
# Define tickers for this ensemble (used throughout notebook)
ensemble_tickers = [Ticker.ES, Ticker.NQ, Ticker.YM, Ticker.RTY]

candles_df = helpers.load_data_multi_ticker(
    tickers=ensemble_tickers,
    timeframe=TimeFrame.D,
    start=datetime(2020, 1, 1),
    end=datetime(2024, 12, 31),
    use_millisecond_offset=True
)

print(f"Candles shape: {candles_df.shape}")
print(f"Candles columns: {list(candles_df.columns)}")
print(f"\nUnique tickers: {candles_df['ticker'].unique()}")
print(f"\nFirst few candles:")
print(candles_df.head())

# Compute targets from candles (for visualization and model fitting)
# We'll compute targets per ticker and include datetime column
targets_list = []
for ticker in candles_df['ticker'].unique():
    ticker_candles = candles_df[candles_df['ticker'] == ticker].copy()
    ticker_targets = pd.DataFrame({
        'datetime': ticker_candles['datetime'].values,
        'raw_return': (ticker_candles['close'] / ticker_candles['open']) - 1,
        'log_return': np.log(ticker_candles['close'] / ticker_candles['open']),
    })
    ticker_targets['ticker'] = ticker
    targets_list.append(ticker_targets)

targets = pd.concat(targets_list, axis=0, ignore_index=True)
print(f"\nTargets shape: {targets.shape}")
print(f"Targets columns: {list(targets.columns)}")

Candles shape: (5040, 9)
Candles columns: ['timestamp', 'datetime', 'open', 'high', 'low', 'close', 'volume', 'ticker', 'timeframe']

Unique tickers: [<Ticker.ES: 'US500'> <Ticker.NQ: 'US100'> <Ticker.YM: 'US30'>
 <Ticker.RTY: 'RUSSELL_2000'>]

First few candles:
    timestamp                datetime      open      high       low     close  \
0  1577923200 2020-01-02 00:00:00.000   3237.00   3261.75   3234.25   3259.00   
1  1577923200 2020-01-02 00:00:00.001   8776.75   8901.50   8769.50   8891.75   
2  1577923200 2020-01-02 00:00:00.002  28553.00  28871.00  28538.00  28841.00   
3  1577923200 2020-01-02 00:00:00.003   1674.00   1684.80   1654.60   1670.50   
4  1578009600 2020-01-03 00:00:00.000   3261.00   3263.50   3206.75   3235.50   

    volume      ticker    timeframe  
0  1416241   Ticker.ES  TimeFrame.D  
1   416637   Ticker.NQ  TimeFrame.D  
2   183583   Ticker.YM  TimeFrame.D  
3   163826  Ticker.RTY  TimeFrame.D  
4  1684630   Ticker.ES  TimeFrame.D  

Targets shape: (5040

In [4]:
# Step 2: Create BaseModel (will compute features internally from candles)
# BaseModel owns bias nodes via composition - no need to pre-extract features!
binning_model = QuantileBinningModel(n_bins=2, selection_metric='sortino', strategy='long')

bias_spec = {
    'module_name': 'buy_hold',
    'timeframes': [TimeFrame.D],
    'params': {}
}

# Create BaseModel for ES
# BaseModel will create bias nodes internally based on bias_node_spec
base_model = BaseModel(
    feature_config={'bias_node_spec': bias_spec},
    binning_model=binning_model,
    ticker=Ticker.ES
)

print("BaseModel created successfully!")
print(f"  Ticker: {base_model.ticker.name}")
print(f"  Bias node spec: {base_model.bias_node_spec}")
print(f"  Number of bias nodes: {len(base_model.bias_nodes)}")
print(f"  Timeframes: {list(base_model.bias_nodes.keys())}")

# Note: Features will be computed internally when we call fit() or add_candle()
# This demonstrates the key design: BaseModel owns bias nodes and computes features from candles

BaseModel created successfully!
  Ticker: ES
  Bias node spec: {'module_name': 'buy_hold', 'timeframes': [<TimeFrame.D: 86400>], 'params': {}}
  Number of bias nodes: 1
  Timeframes: [<TimeFrame.D: 86400>]


In [5]:
# Step 3: Fit BaseModel with candles
# BaseModel.fit() requires candles_df with columns: datetime, open, high, low, close, volume, ticker, timeframe
# BaseModel will compute features internally from candles - no need to pre-extract features!

# Filter candles for ES (the ticker this BaseModel was created for)
es_candles = candles_df[candles_df['ticker'] == Ticker.ES].copy()

# Prepare target data aligned with candles
# BaseModel.fit expects target_data as Series with datetime index
es_targets = targets[targets['ticker'] == Ticker.ES].copy()
es_targets['datetime'] = es_candles['datetime'].values
target_series = pd.Series(
    es_targets['log_return'].values,
    index=pd.DatetimeIndex(es_targets['datetime']),
    name='log_return'
)

# Reset candles index to use integer index (BaseModel.fit expects this)
es_candles_fit = es_candles.reset_index(drop=True)

# Fit model - BaseModel will:
# 1. Stream candles to bias nodes internally (via add_candle)
# 2. Extract features from bias node outputs (via get_feature)
# 3. Fit binning model on features
print("Fitting BaseModel...")
print(f"  Candles: {len(es_candles_fit)} rows")
print(f"  Targets: {len(target_series)} rows")
print(f"  BaseModel will compute features internally from candles")

base_model.fit(es_candles_fit, target_series)

print(f"\n✓ Model fitted successfully!")
print(f"  Feature column: {base_model.feature_column}")
print(f"  Binning model fitted: {base_model.binning_model.is_fitted_}")
print(f"  Best long bin: {base_model.binning_model.best_long_bin_}")
print(f"  Thresholds: {base_model.binning_model.thresholds_}")

# Extract features for visualization (now that model is fitted)
feature_series = base_model.get_feature()
print(f"\nFeature statistics:")
print(f"  Min: {feature_series.min()}")
print(f"  Max: {feature_series.max()}")
print(f"  Mean: {feature_series.mean()}")
print(f"  Unique values: {feature_series.nunique()}")
print(f"  All values == 1.0: {(feature_series == 1.0).all()}")

# Create features DataFrame for FeatureExplorer (for visualization)
features = pd.DataFrame({base_model.feature_column: feature_series})
features['ticker'] = Ticker.ES.name

# Build metadata for FeatureExplorer
metadata = {'feature_metadata': {}}
parsed = helpers.parse_feature_column_name(base_model.feature_column)
metadata['feature_metadata'][base_model.feature_column] = {
    'module': parsed.get('module'),
    'parameters': parsed.get('params', {}),
    'timeframes': [parsed.get('tf')] if parsed.get('tf') else [],
    'base_name': parsed.get('module'),
    'full_name': base_model.feature_column
}

# Create FeatureExplorer for visualization
# Filter ES targets and set datetime as index
es_targets_filtered = targets[targets['ticker'] == Ticker.ES].copy()
es_targets_indexed = es_targets_filtered.set_index('datetime')
# Align targets with feature series index (datetime index from BaseModel)
es_targets_aligned = es_targets_indexed.reindex(feature_series.index)

explorer = FeatureExplorer(
    features_df=features.set_index(feature_series.index),
    targets_df=es_targets_aligned,
    metadata=metadata
)

print("\nFeatureExplorer created successfully!")
print(f"  Number of features: {explorer.n_features}")
print(f"  Feature names: {explorer.feature_names}")
print(f"  Number of samples: {explorer.n_samples}")

Fitting BaseModel...
  Candles: 1260 rows
  Targets: 1260 rows
  BaseModel will compute features internally from candles

✓ Model fitted successfully!
  Feature column: buy_hold_signal_D
  Binning model fitted: True
  Best long bin: 0
  Thresholds: []

Feature statistics:
  Min: 1.0
  Max: 1.0
  Mean: 1.0
  Unique values: 1
  All values == 1.0: True

FeatureExplorer created successfully!
  Number of features: 1
  Feature names: ['buy_hold_signal_D']
  Number of samples: 1260


In [6]:
# Step 4: Visualize signal performance using FeatureExplorer
from metrics.performance import SortinoRatio
metric = SortinoRatio(annualization_factor=252)

figs, summary = explorer.plot_signal_cumsum(
    target_col='log_return',
    base_model=base_model,
    show_plot=True,
    metric=metric,
    strategy='long'
)
plt.show()

print("\nSignal Performance Summary:")
print(summary)


Plotting signal-gated cumulative returns for 1 features
Target: log_return | Strategy: long

[1/1] buy_hold_signal_D
  ✗ Failed: 'Series' object has no attribute 'columns'

Signal Performance Summary:
Empty DataFrame
Columns: []
Index: []


In [7]:
# Step 5: Test predictions (should always be 1 for buy and hold)
test_candles = candles_df.tail(100)  # Last 100 candles
predictions = base_model.predict(test_candles, strategy='long')

print(f"Predictions shape: {predictions.shape}")
print(f"Unique prediction values: {np.unique(predictions)}")
print(f"All predictions == 1: {(predictions == 1).all()}")
print(f"\nFirst 10 predictions:")
print(predictions.head(10))

Predictions shape: (100,)
Unique prediction values: [1]
All predictions == 1: True

First 10 predictions:
2024-11-25 00:00:00.000    1
2024-11-25 00:00:00.001    1
2024-11-25 00:00:00.002    1
2024-11-25 00:00:00.003    1
2024-11-26 00:00:00.000    1
2024-11-26 00:00:00.001    1
2024-11-26 00:00:00.002    1
2024-11-26 00:00:00.003    1
2024-11-27 00:00:00.000    1
2024-11-27 00:00:00.001    1
dtype: int64


In [8]:
# Step 6: Save to vault
print("Initializing vault...")
# Vault root is hardcoded to 'vault' at project root
initialize_vault()

print("Creating ensemble directory...")
# ensemble_tickers is defined in Step 1 - must match the tickers used in data
ensemble_dir = create_ensemble_directory(
    timeframe=TimeFrame.D,
    ensemble_name='buy_hold',
    direction=Direction.LONG,
    tickers=ensemble_tickers
)

print(f"✓ Created ensemble directory: {ensemble_dir}")
print(f"  Ensemble tickers: {[t.name for t in ensemble_tickers]}")
print(f"  Default ensemble directory is now set automatically")

Initializing vault...
Creating ensemble directory...
✓ Created ensemble directory: /home/raman/repos/Trading-Algo/research/vault/D/buy_hold_long
  Ensemble tickers: ['ES', 'NQ', 'YM', 'RTY']
  Default ensemble directory is now set automatically


In [9]:

print("\nSaving base model to vault...")
# Pass all ensemble tickers when saving (even though BaseModel was created for ES)
# This ensures the feature spec stores all tickers the ensemble uses
# ensemble_dir is optional - uses default set by create_ensemble_directory()
model_id = base_model.save_to_vault(tickers=ensemble_tickers)
print(f"✓ Saved model with ID: {model_id}")
print(f"  Feature column: {base_model.feature_column}")
print(f"  Model ID: {model_id}")
print(f"  Saved with tickers: {[t.name for t in ensemble_tickers]}")


Saving base model to vault...
✓ Saved model with ID: quantile_binning_2
  Feature column: buy_hold_signal_D
  Model ID: quantile_binning_2
  Saved with tickers: ['ES', 'NQ', 'YM', 'RTY']


In [10]:
# Step 7: Verify vault save/load
print("Loading models from vault...")
# ensemble_dir is optional - uses default set by create_ensemble_directory()
# Path resolution automatically handles running from research/ directory
loaded_models = load_feature_base_models(feature_column=base_model.feature_column)
model_id = 'quantile_binning_2'

print(f"✓ Loaded {len(loaded_models)} model(s)")
print(f"  Model keys (ticker, model_id): {list(loaded_models.keys())}")

# Verify loaded model (keys are now (ticker, model_id) tuples)
key = (Ticker.ES, model_id)
if key in loaded_models:
    loaded_model = loaded_models[key]
    print(f"\n✓ Model '{model_id}' for {Ticker.ES.name} loaded successfully")
    print(f"  Binning model fitted: {loaded_model.binning_model.is_fitted_}")
    print(f"  Best long bin: {loaded_model.binning_model.best_long_bin_}")
    print(f"  Ticker: {loaded_model.ticker.name}")
    
    # Test predictions with loaded model
    test_predictions = loaded_model.predict(test_candles, strategy='long')
    print(f"  Predictions from loaded model all == 1: {(test_predictions == 1).all()}")
else:
    print(f"\n✗ Model '{model_id}' not found. Available keys: {list(loaded_models.keys())}")

# List all features in ensemble
print("\n" + "="*70)
print("Features in ensemble:")
print("="*70)
# ensemble_dir is optional - uses default set by create_ensemble_directory()
features_df = list_features()
print(features_df)

Loading models from vault...
✓ Loaded 4 model(s)
  Model keys (ticker, model_id): [(<Ticker.ES: 'US500'>, 'quantile_binning_2'), (<Ticker.NQ: 'US100'>, 'quantile_binning_2'), (<Ticker.RTY: 'RUSSELL_2000'>, 'quantile_binning_2'), (<Ticker.YM: 'US30'>, 'quantile_binning_2')]

✓ Model 'quantile_binning_2' for ES loaded successfully
  Binning model fitted: True
  Best long bin: 0
  Ticker: ES
  Predictions from loaded model all == 1: True

Features in ensemble:
        feature_name     feature_column  n_base_models  n_fitted  \
0  buy_hold_signal_D  buy_hold_signal_D              1         1   

                         created_at                        updated_at  
0  2026-01-23T04:04:43.351394+00:00  2026-01-23T04:04:43.351415+00:00  
